# Data Acquisition from Materials Databases

This notebook retrieves materials science datasets from external APIs and databases.

## Sources
- Materials Project API (requires API key)
- JARVIS Database
- Other materials science repositories

In [37]:
# Load environment variables
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# scikit-learn
from pathlib import Path
from dotenv import load_dotenv
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Matminer — dataset loader and featurizers
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import StructureToComposition
from matminer.featurizers.base import MultipleFeaturizer
from pymatgen.core import Composition
import typing
from typing_extensions import NotRequired
typing.NotRequired = NotRequired

# Set up paths
PROJECT_ROOT = Path().cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

# Materials Project API Configuration
MATERIALS_PROJECT_API_KEY=ch4cGSRTN5xQL0X8KJJOxsCCmbrc076x

print('✅ All imports successful')

Task was destroyed but it is pending!
task: <Task pending name='Task-1459' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-1460' coro=<_async_in_context.<locals>.preserve_context() running at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-1460' coro=<_async_in_context.<locals>.preserve_context() running at /opt/anaconda3/envs/matds/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]>


NameError: name 'ch4cGSRTN5xQL0X8KJJOxsCCmbrc076x' is not defined

In [12]:
# Query Materials Project for binary carbon compounds with transition metals
from mp_api.client import MPRester
import os

# Get API key from environment
api_key = os.getenv('MP_API_KEY')
if not api_key:
    raise ValueError("MP_API_KEY not found in .env file. Get one at https://materialsproject.org/api")

# Initialize Materials Project client
mpr = MPRester(api_key=api_key)
print("Connected to Materials Project API")

Connected to Materials Project API


In [27]:
# Define transition metals (d-block elements)
transition_metals = [
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",  # 3d
    "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",  # 4d
    "La", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg"   # 5d
]

def is_binary_tm_o(elements_list, tm):
    """Check if compound has exactly 2 elements: O and the specified transition metal"""
    if len(elements_list) != 2:
        return False
    
    # Get element symbols as strings
    element_symbols = set([str(elem).replace('Element ', '') for elem in elements_list])
    
    # Should have exactly O and the transition metal
    return element_symbols == {tm, 'O'}

# Query for binary transition metal-oxygen compounds
tm_o_binary = []

for tm in transition_metals:
    print(f"Querying {tm}-O binary compounds...")
    try:
        materials = mpr.summary.search(
            elements=[tm, "O"],
            fields=[
                "material_id", 
                "formula_pretty", 
                "elements",
                "energy_above_hull", 
                "volume", 
                "density",
                "band_gap",
                "is_stable",
                "is_metal"
            ]
        )
        
        binary_found = 0
        if materials:
            for mat in materials:
                # Only keep if it's truly binary (exactly 2 elements: O and TM)
                if is_binary_tm_o(mat.elements, tm):
                    compound_data = {
                        'material_id': mat.material_id,
                        'formula': mat.formula_pretty,
                        'transition_metal': tm,
                        'energy_above_hull': mat.energy_above_hull,
                        'volume': mat.volume,
                        'density': mat.density,
                        'band_gap': mat.band_gap,
                        'is_stable': mat.is_stable,
                        'is_metal': mat.is_metal,
                    }
                    tm_o_binary.append(compound_data)
                    binary_found += 1
            print(f"  Found {binary_found} binary compounds (out of {len(materials)} total)")
    except Exception as e:
        print(f"Error querying {tm}-O: {e}")

print(f"\nTotal BINARY TM-O compounds found: {len(tm_o_binary)}")

# Save the filtered dataset
df_binary = pd.DataFrame(tm_o_binary)
output_file = DATA_DIR / 'tm_oxygen_binary_compounds.csv'
df_binary.to_csv(output_file, index=False)
print(f"Saved {len(df_binary)} binary compounds to {output_file}")

# Show summary
print("\nBinary TM-O Dataset Summary:")
print(f"Shape: {df_binary.shape}")
print(f"Columns: {list(df_binary.columns)}")
print("\nFirst 10 rows:")
print(df_binary.head(10))
print("\nTransition metals found:")
print(df_binary['transition_metal'].value_counts())

Querying Sc-O binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/1044 [00:00<?, ?it/s]

  Found 13 binary compounds (out of 1044 total)
Querying Ti-O binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/5283 [00:00<?, ?it/s]

  Found 124 binary compounds (out of 5283 total)
Querying V-O binary compounds...


...

  Found 15 binary compounds (out of 389 total)
Querying Au-O binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/250 [00:00<?, ?it/s]

  Found 3 binary compounds (out of 250 total)
Querying Hg-O binary compounds...


Retrieving SummaryDoc documents:   0%|          | 0/534 [00:00<?, ?it/s]

  Found 9 binary compounds (out of 534 total)

Total BINARY TM-O compounds found: 1223
Saved 1223 binary compounds to /Users/benbenmerk/Merkin_FinalProject/data/tm_oxygen_binary_compounds.csv

Binary TM-O Dataset Summary:
Shape: (1223, 9)
Columns: ['material_id', 'formula', 'transition_metal', 'energy_above_hull', 'volume', 'density', 'band_gap', 'is_stable', 'is_metal']

First 10 rows:
  material_id formula transition_metal  energy_above_hull      volume  \
0  mp-1179108    ScO2               Sc           0.328212   77.525876   
1  mp-1179114    ScO2               Sc           0.272273  176.553978   
2  mp-1186987    ScO3               Sc           0.692938   75.201810   
3  mp-1206234    ScO2               Sc           0.274362   72.282358   
4   mp-644481     ScO               Sc           0.066787   22.131499   
5   mp-775837   Sc2O3               Sc           0.081780  506.463681   
6      mp-216   Sc2O3               Sc           0.000000  479.729109   
7    mp-13060   Sc2O3     

In [31]:
# Convert to DataFrame first
df_raw = pd.DataFrame(tm_o_binary)

print("Raw dataset shape:", df_raw.shape)
print("\nMissing values before cleaning:")
print(df_raw.isnull().sum())

# === FILTERING ===

# 1. Remove duplicates (same formula)
df_cleaned = df_raw.drop_duplicates(subset=['formula'], keep='first')
print(f"\nAfter removing duplicates: {len(df_cleaned)} compounds (removed {len(df_raw) - len(df_cleaned)})")

# 2. Remove compounds with missing critical properties
df_cleaned = df_cleaned.dropna(subset=['volume', 'density'])
print(f"After removing missing volume/density: {len(df_cleaned)} compounds")

# 3. Remove entries with invalid values (e.g., negative volume)
df_cleaned = df_cleaned[df_cleaned['volume'] > 0]
df_cleaned = df_cleaned[df_cleaned['density'] > 0]
print(f"After removing invalid values: {len(df_cleaned)} compounds")

# 4. Filter by energy above hull (optional - keep only relatively stable compounds)
# Keep compounds within 500 meV/atom of the convex hull
df_cleaned = df_cleaned[df_cleaned['energy_above_hull'] < 1]  # 1 eV/atom threshold
print(f"After filtering by energy_above_hull < 1 eV: {len(df_cleaned)} compounds")

# === CLEANING ===

# 1. Handle missing band gap values (some materials don't have this calculated)
df_cleaned['band_gap'] = df_cleaned['band_gap'].fillna(-1)  # -1 indicates missing
print(f"\nFilled missing band_gap values with -1")

# 2. Ensure is_stable is boolean
df_cleaned['is_stable'] = df_cleaned['is_stable'].astype(bool)
df_cleaned['is_metal'] = df_cleaned['is_metal'].astype(bool)

# 3. Add a compound count column
df_cleaned['num_materials'] = 1

# === DATA VALIDATION ===
print("\n=== Cleaned Dataset Summary ===")
print(f"Final shape: {df_cleaned.shape}")
print(f"\nData types:")
print(df_cleaned.dtypes)
print(f"\nMissing values after cleaning:")
print(df_cleaned.isnull().sum())
print(f"\nBasic statistics:")
print(df_cleaned.describe())

print(f"\nStability distribution:")
print(f"  Stable: {df_cleaned['is_stable'].sum()}")
print(f"  Unstable: {(~df_cleaned['is_stable']).sum()}")

print(f"\nMetal vs Non-metal:")
print(f"  Metals: {df_cleaned['is_metal'].sum()}")
print(f"  Non-metals: {(~df_cleaned['is_metal']).sum()}")

Raw dataset shape: (1223, 9)

Missing values before cleaning:
material_id          0
formula              0
transition_metal     0
energy_above_hull    0
volume               0
density              0
band_gap             0
is_stable            0
is_metal             1
dtype: int64

After removing duplicates: 306 compounds (removed 917)
After removing missing volume/density: 306 compounds
After removing invalid values: 306 compounds
After filtering by energy_above_hull < 1 eV: 288 compounds

Filled missing band_gap values with -1

=== Cleaned Dataset Summary ===
Final shape: (288, 10)

Data types:
material_id           object
formula               object
transition_metal      object
energy_above_hull    float64
volume               float64
density              float64
band_gap             float64
is_stable               bool
is_metal                bool
num_materials          int64
dtype: object

Missing values after cleaning:
material_id          0
formula              0
transition_met

In [35]:
# Save the cleaned dataset
output_file = DATA_DIR / 'tm_oxygen_binary_compounds.csv'
df_cleaned.to_csv(output_file, index=False)
print(f"Saved cleaned dataset to {output_file}")
print(f"Final dataset: {len(df_cleaned)} compounds with {len(df_cleaned.columns)} columns")

# Also save a data quality report
report_file = DATA_DIR / 'data_quality_report.txt'
with open(report_file, 'w') as f:
    f.write("=== TM-O Binary Compounds Data Quality Report ===\n\n")
    f.write(f"Total compounds acquired: {len(df_raw)}\n")
    f.write(f"After deduplication: {len(df_raw.drop_duplicates(subset=['formula']))}\n")
    f.write(f"After cleaning & filtering: {len(df_cleaned)}\n")
    f.write(f"\nFiltering criteria applied:\n")
    f.write(f"  - Removed duplicates by formula\n")
    f.write(f"  - Removed missing volume/density\n")
    f.write(f"  - Kept only positive volume and density\n")
    f.write(f"  - Energy above hull < 0.5 eV/atom\n")
    f.write(f"\nFinal dataset statistics:\n")
    f.write(f"  Stable compounds: {df_cleaned['is_stable'].sum()}\n")
    f.write(f"  Metals: {df_cleaned['is_metal'].sum()}\n")
    f.write(f"  Transition metals represented: {df_cleaned['transition_metal'].nunique()}\n")

print(f"Saved data quality report to {report_file}")

# Also save locally to data/ folder
output_file = DATA_DIR / 'tm_oxygen_binary_compounds.csv'
df_cleaned.to_csv(output_file, index=False)
print(f"\nSaved cleaned dataset to {output_file}")

Saved cleaned dataset to /Users/benbenmerk/Merkin_FinalProject/data/tm_oxygen_binary_compounds.csv
Final dataset: 288 compounds with 10 columns
Saved data quality report to /Users/benbenmerk/Merkin_FinalProject/data/data_quality_report.txt

Saved cleaned dataset to /Users/benbenmerk/Merkin_FinalProject/data/tm_oxygen_binary_compounds.csv


## TODO: Add data acquisition code
- Connect to Materials Project API
- Query for materials with specific properties
- Handle pagination and rate limits
- Save to CSV